# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)

Schema URL: [`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Name: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), fields (`cr:Field`), and their `@id` values.
Below we will inspect what record sets and fields are defined in the dataset.

In [ ]:
# List all record sets and their fields from dataset metadata
for rs in metadata.record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"@id          : {rs.id}")
    field_ids = [field.id for field in rs.fields]
    print(f"Fields       : {field_ids}\n")

You can also inspect the first record from a record set for more detail. Let's print out one record from the main data record set (replace with specific `@id` if needed):

In [ ]:
# Example: Print a record from the main record set
# You will need to copy the actual @id of the main data RecordSet from the above cell output.

main_record_set_id = None
for rs in metadata.record_sets:
    if 'protein' in (rs.name or '').lower() or 'main' in (rs.name or '').lower():
        main_record_set_id = rs.id
        break
# Fallback to first record set if none matched
if main_record_set_id is None and metadata.record_sets:
    main_record_set_id = metadata.record_sets[0].id
if main_record_set_id:
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        print(f"Record example from '{main_record_set_id}':\n", rec)
        if i >= 0:
            break

## 3. Data Extraction
Load data from one or more record sets into a DataFrame for analysis.
All entities (record sets, fields, columns) are referenced by their `@id` values.

In [ ]:
# Prepare extraction of all available record sets
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from RecordSet {rs_id}\nColumns: {df.columns.tolist()[:10]}{'...' if len(df.columns) > 10 else ''}\n")
# Use the main record set above (or change as needed)
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Field @ids in DataFrame for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, transforming columns, normalizing numeric fields, and basic grouping.

Below, select a numeric field and a group/categorical field by their `@id`, as shown in the previous overview.

In [ ]:
# Example: Identify likely numeric and group field IDs (copy actual @ids from previous output)

# Try to select standard proteomics fields; adjust as needed
# e.g., '@id': 'cr:peptideCount' and group on '@id': 'cr:description'

record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Guess a numeric field id (adjust as needed)
numeric_field_id = None
for col in df.columns:
    if 'count' in col.lower() or 'abundance' in col.lower() or 'coverage' in col.lower() or 'mw' in col.lower() or 'mass' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: Select first float-like column
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

group_field_id = None
for col in df.columns:
    if col != numeric_field_id and ('desc' in col.lower() or 'type' in col.lower() or 'group' in col.lower() or 'category' in col.lower()):
        group_field_id = col
        break

print(f"Selected numeric field: {numeric_field_id}")
print(f"Selected group field:   {group_field_id}")

if numeric_field_id is not None and numeric_field_id in df.columns:
    # Attempt filtering on numeric value (>10)
    threshold = 10
    numeric_vals = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[numeric_vals > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
    display(filtered_df.head())
    # Normalize the column
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()) / filtered_df[numeric_field_id].astype(float).std()
    print(f"First 5 records with normalized '{numeric_field_id}':")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No suitable numeric field found for EDA.")

# Optionally group by a field and aggregate normalized value
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[f"{numeric_field_id}_normalized"].mean().reset_index()
    print(f"Grouped average normalized '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())

## 5. Visualization
Visualize distributions and relationships in the dataset using matplotlib and seaborn.

Plots below use field `@id`s for full transparency.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=40, kde=True, color='steelblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    if group_field_id and group_field_id in df.columns:
        topcats = df[group_field_id].value_counts().index[:5]
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df[df[group_field_id].isin(topcats)], x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (top 5 groups)")
        plt.ylabel(numeric_field_id)
        plt.xlabel(group_field_id)
        plt.show()


## 6. Conclusion
In this notebook, we've demonstrated how to access, load, and analyze the FAIR<sup>2</sup> Mass Spectrometry dataset using the `mlcroissant` library.

- The dataset is structured via the Croissant schema, enabling robust and transparent referencing via `@id`s.
- We loaded all available record sets, inspected fields, and performed exploratory analysis using referenced field IDs.
- Visualizations highlighted numeric field distributions, and we demonstrated normalization and grouping by categorical attributes.

For custom analysis, always check field and record set IDs, and adapt the analysis accordingly.

**Dataset DOI**: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)